# Pipeline Đếm Hạt Gạo

Notebook này dùng trực tiếp code trong `src/`. Toàn bộ tham số lấy từ `DEFAULT_CONFIG`, vì vậy khi sửa `src/config.py`, notebook sẽ tự động dùng cùng cấu hình với script.


## 0. Import code từ `src`

Cell này thêm thư mục `src/` vào `sys.path`, sau đó import các hàm xử lý chính. Notebook không định nghĩa lại pipeline để tránh lệch logic hoặc lệch tham số.


In [53]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from skimage import color, measure

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import DEFAULT_CONFIG
from main import run_all_images
from preprocess import (
    correct_illumination,
    denoise_image,
    enhance_contrast,
    preprocess_image,
    to_grayscale,
)
from segment import clean_mask, segment_grains, threshold_grains
from utils import list_image_files, read_image
from visualize import contour_overlay
from watershed_count import filter_grain_regions, separate_and_count, watershed_separation

CONFIG = DEFAULT_CONFIG
DATASET_DIR = CONFIG.dataset_dir
OUTPUT_DIR = CONFIG.output_dir
LOG_DIR = CONFIG.log_dir

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.titlesize"] = 11

print("Project root:", PROJECT_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Output dir:", OUTPUT_DIR)
print("Log dir:", LOG_DIR)
print("Config:", CONFIG)


Project root: c:\Users\Admin\Desktop\Cao học Bách Khoa\Thị giác máy tính\Assigment\Computer-Vision
Dataset dir: Dataset
Output dir: output
Log dir: logs
Config: PipelineConfig(dataset_dir=WindowsPath('Dataset'), output_dir=WindowsPath('output'), log_dir=WindowsPath('logs'), median_kernel_size=3, background_kernel_size=51, clahe_clip_limit=2.0, clahe_tile_grid_size=(8, 8), morphology_kernel_size=3, adaptive_block_size=51, adaptive_c=-5, min_grain_area=60, max_grain_area=8000, min_aspect_ratio=1.2, max_aspect_ratio=10.0, min_solidity=0.45, min_peak_distance=14)


## 1. Hàm tiện ích hiển thị

Các hàm dưới đây chỉ dùng để in thống kê và hiển thị ảnh trong notebook. Chúng không thay đổi pipeline xử lý ảnh.


In [54]:
def image_summary(name, image):
    array = np.asarray(image)
    return {
        "name": name,
        "shape": array.shape,
        "dtype": str(array.dtype),
        "min": float(array.min()),
        "max": float(array.max()),
        "mean": float(array.mean()),
        "std": float(array.std()),
    }


def mask_summary(name, mask):
    labels = measure.label(mask.astype(bool))
    return {
        "name": name,
        "foreground_pixels": int(mask.sum()),
        "foreground_ratio": float(mask.mean()),
        "connected_components": int(labels.max()),
    }


def label_summary(name, labels):
    return {
        "name": name,
        "label_count": int(labels.max()),
        "labeled_pixels": int((labels > 0).sum()),
    }


def print_image_result(name, image):
    print(f"[{name}]")
    display(pd.DataFrame([image_summary(name, image)]))


def print_mask_result(name, mask):
    print(f"[{name}]")
    display(pd.DataFrame([mask_summary(name, mask)]))


def print_label_result(name, labels):
    print(f"[{name}]")
    display(pd.DataFrame([label_summary(name, labels)]))


def show_images(items, cols=3):
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for axis, (title, image) in zip(axes, items):
        axis.set_title(title)
        if image.ndim == 2:
            axis.imshow(image, cmap="gray")
        else:
            axis.imshow(image)
        axis.axis("off")

    for axis in axes[len(items):]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()


def show_histogram(title, image):
    plt.figure(figsize=(8, 4))
    plt.hist(np.asarray(image).ravel(), bins=256, color="steelblue")
    plt.title(title)
    plt.xlabel("Intensity")
    plt.ylabel("Frequency")
    plt.show()


## 2. Đọc ảnh và kiểm tra dataset

Sử dụng `list_image_files` và `read_image` từ `src/utils.py`. Sau khi đọc ảnh, notebook in thống kê và hiển thị toàn bộ ảnh đầu vào.


In [55]:
image_paths = list_image_files(DATASET_DIR)
print("[list_image_files]")
print(f"Tìm thấy {len(image_paths)} ảnh")
for index, path in enumerate(image_paths):
    print(index, path.name)

input_images = [(path.name, read_image(path)) for path in image_paths]
show_images(input_images, cols=2)

SAMPLE_NAME = "gạo_bình_thường.png"
sample_path = next(path for path in image_paths if path.name == SAMPLE_NAME)
sample_image = read_image(sample_path)
print_image_result("read_image - sample image", sample_image)
print("Ảnh mẫu:", sample_path.name, sample_image.shape)


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'Dataset'

## 3. Tiền xử lý ảnh

Các bước tiền xử lý đều gọi trực tiếp từ `src/preprocess.py`: grayscale, median blur, hiệu chỉnh nền và CLAHE.


In [ ]:
print_image_result("input image", sample_image)

gray = to_grayscale(sample_image)
print_image_result("to_grayscale", gray)

denoised = denoise_image(gray, CONFIG.median_kernel_size)
print_image_result("denoise_image", denoised)

corrected = correct_illumination(denoised, CONFIG.background_kernel_size)
print_image_result("correct_illumination", corrected)

enhanced = enhance_contrast(corrected, CONFIG)
print_image_result("enhance_contrast", enhanced)

preprocessed = preprocess_image(sample_image, CONFIG)
print_image_result("preprocess_image", preprocessed)
print("preprocess_image equals step-by-step enhanced:", bool(np.array_equal(preprocessed, enhanced)))

show_images([
    ("original", sample_image),
    ("gray", gray),
    ("denoised", denoised),
    ("illumination corrected", corrected),
    ("CLAHE enhanced", enhanced),
    ("preprocess_image output", preprocessed),
], cols=3)

show_histogram("Histogram - grayscale", gray)
show_histogram("Histogram - enhanced", enhanced)


## 4. Phân đoạn hạt gạo

Các bước phân đoạn gọi từ `src/segment.py`. `threshold_grains` dùng Otsu với adaptive fallback tự động, sau đó `clean_mask` loại nhiễu nhỏ và làm sạch vùng hạt.


In [ ]:
raw_mask = threshold_grains(enhanced, CONFIG)
print_mask_result("threshold_grains", raw_mask)

cleaned_mask = clean_mask(raw_mask, CONFIG)
print_mask_result("clean_mask", cleaned_mask)

segmented_mask = segment_grains(enhanced, CONFIG)
print_mask_result("segment_grains", segmented_mask)
print("segment_grains equals clean_mask(threshold_grains):", bool(np.array_equal(segmented_mask, cleaned_mask)))

show_images([
    ("enhanced", enhanced),
    ("raw mask", raw_mask),
    ("cleaned mask", cleaned_mask),
    ("segment_grains output", segmented_mask),
], cols=2)


## 5. Tách hạt dính nhau và đếm

Các bước watershed và lọc vùng gọi từ `src/watershed_count.py`. Kết quả cuối cùng được so sánh với hàm `separate_and_count` dùng trong script.


In [ ]:
labels = watershed_separation(cleaned_mask, CONFIG)
print_label_result("watershed_separation", labels)

filtered_labels, rejected_regions = filter_grain_regions(labels, CONFIG)
print_label_result("filter_grain_regions", filtered_labels)
print("rejected regions:", rejected_regions)

count_result = separate_and_count(SAMPLE_NAME, cleaned_mask, CONFIG)
print("[separate_and_count]")
print("  image:", count_result.image_name)
print("  final count:", count_result.count)
print("  rejected regions:", count_result.rejected_regions)

show_images([
    ("cleaned mask", cleaned_mask),
    ("watershed labels", color.label2rgb(labels, bg_label=0)),
    ("filtered labels", color.label2rgb(filtered_labels, bg_label=0)),
    ("contour overlay", contour_overlay(sample_image, count_result.labels)),
], cols=2)


## 6. Chạy toàn bộ dataset

Cell này gọi `run_all_images()` từ `src/main.py`. Hàm này dùng cùng pipeline, cùng `DEFAULT_CONFIG`, lưu `output/results.csv` và ghi log local trong `logs/`.


In [ ]:
batch_results = run_all_images()
results_df = pd.DataFrame([
    {
        "image_name": result.image_name,
        "count": result.count,
        "rejected_regions": result.rejected_regions,
    }
    for result in batch_results
])

display(results_df)
print("Saved:", OUTPUT_DIR / "results.csv")
print("Log updated:", LOG_DIR / "parameter_runs.md")


## 7. Kiểm tra contour overlay

Cell này hiển thị contour overlay cho từng ảnh để đánh giá trực quan. Khi đánh giá bắt buộc giữ nguyên `DEFAULT_CONFIG` cho toàn bộ ảnh, không chỉnh tham số riêng theo từng ảnh.


In [ ]:
overlays = []
for path in image_paths:
    image = read_image(path)
    mask = segment_grains(preprocess_image(image, CONFIG), CONFIG)
    result = separate_and_count(path.name, mask, CONFIG)
    overlay = contour_overlay(image, result.labels)
    overlays.append((f"{path.name} | count={result.count}", overlay))

show_images(overlays, cols=2)


## 8. Ghi chú

Notebook này không giữ bản sao tham số riêng. Muốn điều chỉnh kết quả, sửa một bộ tham số chung trong `src/config.py`, chạy lại toàn bộ ảnh, rồi xem log trong `logs/parameter_runs.md`.
